# Basic Python script for Tabular ML


Virtual environment sanity check

In [2]:
import sys; print(sys.executable); print(sys.version)

/Users/jonathangoings/Documents/GIT/py-ml-training/.venv/bin/python
3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]


Import packages that will be used

In [3]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Load the built-in breast cancer dataset. Inspect its shape, columns and class balance

In [4]:
dataset = load_breast_cancer(as_frame=True)
df: pd.DataFrame = dataset.frame.copy()
df["target"] = dataset.target

print("Shape (rows, columns):", df.shape)
print("\nTarget counts:")
print(df["target"].value_counts())

df.head()

Shape (rows, columns): (569, 31)

Target counts:
target
1    357
0    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


Prep the data.

- X = the features to use
- Y = the answer to predict

Separate training and test data, setting aside 20% of the data for testing (`test_size=.2`), and make sure the training/tests sets have roughly the same amount of answers (`stratify=y`)

In [5]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train shape (rows, features):", X_train.shape)
print("X_test shape (rows, features):", X_test.shape)
print("y_train class proportions:\n", y_train.value_counts(normalize=True).round(3))
print("y_test class proportions:\n", y_test.value_counts(normalize=True).round(3))

X_train shape (rows, features): (455, 30)
X_test shape (rows, features): (114, 30)
y_train class proportions:
 target
1    0.626
0    0.374
Name: proportion, dtype: float64
y_test class proportions:
 target
1    0.632
0    0.368
Name: proportion, dtype: float64


Build the model

The Pipeline is setup to run the model consistently.

- StandardScaler() shifts each feature so its mean is roughly 0 and its standard deviation is 1 (based only on X_train)
- LogisticRegression() looks at the feature values (X_train) and the known answers (y_train) and adjusts internal weights to learn which patterns point toward each class.

`model.fit` calls the pipeline for the passed variables.

In [6]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


The model has been trained. Run it against the test data and do the comparisons.

In [10]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")

print("Confusion Matrix")
cm = confusion_matrix(y_test, y_pred)
labels = list(dataset.target_names)  # ['malignant', 'benign']

cm_df = pd.DataFrame(
    cm,
    index=[f"Actually {name}" for name in labels],
    columns=[f"Guessed {name}" for name in labels],
)

print(cm_df.to_string())

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=list(dataset.target_names)))

Test Accuracy: 0.9825

Confusion Matrix
                    Guessed malignant  Guessed benign
Actually malignant                 41               1
Actually benign                     1              71

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



Display probabilities for benign class (1 in the dataset). `[:, 1]`

By default, model.predict(X_test) uses a 0.5 threshold:

- If prob_class_1 >= 0.5, predicted_label is 1 (benign).
- If prob_class_1 < 0.5, predicted_label is 0 (malignant).

Comparing actual_label, predicted_label, and prob_class_1 side by side lets you see whether a prediction was confident or not-so

In [12]:
probabilities = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame(
    {
        "actual_answer": y_test.values,
        "predicted_answer": y_pred,
        "prob_class_1": probabilities.round(4),
    },
    index=X_test.index,
)

results.head(10)

,actual_answer,predicted_answer,prob_class_1
256,0,0,0.0000
428,1,1,1.0000
501,0,0,0.0064
363,1,1,0.5335
564,0,0,0.0000
464,1,1,0.9922
358,1,1,1.0000
343,0,0,0.0000
516,0,0,0.0001
567,0,0,0.0000
